# UK 05 · 特征概览（land use · Built-S · C/U/Z · NTL）

只读 `04_features.py` 落地的四类 NPZ（要求 `uk.features.derive` 已 DONE，否则 fail-closed）。展示来源与许可、数组 schema、逐区域覆盖统计、时点，以及标准图。

In [ ]:
from sglib.core.infra.paths import find_repo_root
from sglib.dataoverview import stage
from sglib.dataoverview.overview import figures, tables
from sglib.dataoverview.reader import canonical_paths, read_artifact
import pandas as pd

ROOT = find_repo_root()
COUNTRY = "uk"
config = stage.country_config(ROOT, COUNTRY)
context = stage.country_context(ROOT, COUNTRY)
FIGURES = stage.figures_root(ROOT, COUNTRY)
LABEL = config.country_profile.label
GROUP_KEY = config.values["regions"].get("group_key")
GROUPS = [str(item["id"]) for item in config.values["regions"]["items"]]
contract = config.values["station_contract"]
SOURCE_KEY = str(contract["source_key"])
REGION_COLUMN = str(contract["region_column"])

import numpy as np
from sglib.dataoverview.processing.features.grid_bundle import load_grid_bundle
from sglib.dataoverview.processing.features.cuz_support import load_cuz_support
stage.require_done(ROOT, COUNTRY, ["features"])

## 1 · 来源与许可（source_and_license）

OSM 快照与 VIIRS NTL 是外部数据集；GHSL Built-S 与 C/U/Z 是共享处理产物。

In [ ]:
tables.dataset_ledger(config.values, categories=["landuse", "ntl"])

In [ ]:
tables.product_ledger(config.values, products=["features"])

In [ ]:
pd.DataFrame([
    {"feature": name, **{key: str(value) for key, value in spec.items() if key != "sectors"}}
    for name, spec in config.values["features"].items()
])

## 2 · Schema

合同在 `../schemas/{landuse,built_surface,cuz_support,ntl}.toml`；下面是首个分析区域四个 NPZ 的实际键。

In [ ]:
REGION = GROUPS[0]
ARTIFACTS = {
    "landuse": context.artifact_root / f"{REGION}_landuse.npz",
    "built_surface": context.artifact_root / f"{REGION}_ghsl_built_s.npz",
    "cuz_support": context.artifact_root / f"{REGION}_cuz_support.npz",
    "ntl": context.artifact_root / f"{REGION}_ntl.npz",
}
tables.npz_schema(ARTIFACTS)

## 3 · 覆盖统计（coverage_stats）

C/U/Z 三掩码恰好划分每个区域的所有单元；Z 单元的特征必须严格为零（`zero_nonzero_values == 0`）。

In [ ]:
coverage = tables.feature_coverage(context)
assert (coverage["zero_nonzero_values"] == 0).all(), "Z exact-zero invariant failed"
coverage

## 4 · 单位与时点（unit_and_vintage）

In [ ]:
tables.unit_and_vintage(config).query("section == 'temporal'")

In [ ]:
pd.Series({key: str(value) for key, value in config.values["ntl"].items()}, name="value").to_frame()

## 5 · 标准图（standard_figures）

In [ ]:
LU_COLUMNS = [column for column in coverage.columns if column.startswith("mean_lu_")]
LU_LABELS = [column.removeprefix("mean_lu_").removesuffix("_prop") for column in LU_COLUMNS]
fig = figures.plot_stacked_shares(coverage, index_column="region", columns=LU_COLUMNS, labels=LU_LABELS,
                                  colors=figures.LANDUSE_COLORS, title=f"{LABEL}: mean OSM land-use shares per analysis region", xlabel="share of cell area")
figures.save_figure(fig, FIGURES / "05_landuse_shares.png")
fig

In [ ]:
fig = figures.plot_stacked_shares(coverage, index_column="region", columns=["covered", "unknown", "zero"],
                                  colors=figures.CUZ_COLORS, title=f"{LABEL}: C/U/Z support partition per analysis region", xlabel="share of cells")
figures.save_figure(fig, FIGURES / "05_cuz_partition.png")
fig

单区域地图（默认 `GROUPS[0]`）：主导土地利用类别、C/U/Z 划分、Built-S 比例、NTL 辐射。

In [ ]:
grid, step, metadata = load_grid_bundle(REGION, context.grid_root)
with np.load(ARTIFACTS["landuse"], allow_pickle=False) as archive:
    lu_names = [str(c).removeprefix("lu_").removesuffix("_prop") for c in archive["columns"]]
    lu_data = archive["data"]
dominant = np.where(lu_data.sum(axis=1) > 0, np.array(lu_names)[lu_data.argmax(axis=1)], "no_osm_landuse")
fig = figures.plot_cell_categories(grid, dominant, title=f"{REGION}: dominant OSM land use per cell",
                                   colors={**figures.LANDUSE_COLORS, "no_osm_landuse": figures.GREY_5}, order=[*lu_names, "no_osm_landuse"])
figures.save_figure(fig, FIGURES / f"05_landuse_dominant_{REGION}.png")
fig

In [ ]:
support = load_cuz_support(ARTIFACTS["cuz_support"])
cuz = np.where(support.covered_mask, "covered", np.where(support.unknown_mask, "unknown", "zero"))
fig = figures.plot_cell_categories(grid, cuz, title=f"{REGION}: C/U/Z support partition per cell",
                                   colors=figures.CUZ_COLORS, order=["covered", "unknown", "zero"])
figures.save_figure(fig, FIGURES / f"05_cuz_{REGION}.png")
fig

In [ ]:
with np.load(ARTIFACTS["built_surface"], allow_pickle=False) as archive:
    built = archive["data"][:, 0]
with np.load(ARTIFACTS["ntl"], allow_pickle=False) as archive:
    ntl = archive["data"][:, 0]
fig = figures.plot_cell_values(grid, built, title=f"{REGION}: GHSL Built-S fraction per cell", label="built fraction", cmap="magma")
figures.save_figure(fig, FIGURES / f"05_built_fraction_{REGION}.png")
fig

In [ ]:
fig = figures.plot_cell_values(grid, ntl, title=f"{REGION}: VIIRS night-time light per cell", label="radiance", log=True, cmap="cividis")
figures.save_figure(fig, FIGURES / f"05_ntl_{REGION}.png")
fig

In [ ]:
fig = figures.plot_value_histograms({"ghsl_built_fraction": built, "ntl_value": ntl}, title=f"{REGION}: per-cell value distributions", log_x=True)
figures.save_figure(fig, FIGURES / f"05_feature_histograms_{REGION}.png")
fig

下一步：`06_inventory.ipynb`。